**Import libraries and set the plot style**

Import pandas for tables, NumPy for numerical operations, and Seaborn, Matplotlib, Squarify, and Plotly Express for visualizations. The aliases (`pd`, `np`, `sns`, `plt`, and `px`) shorten later calls. `%matplotlib inline` displays Matplotlib plots inside the notebook, and `whitegrid` adds a grid to Seaborn plots. Some imported libraries are not used in the cells shown below.


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import squarify
import numpy as np
import plotly.express as px

# To display plots directly in the notebook:
%matplotlib inline

# Setting a visual style for seaborn:
sns.set_style('whitegrid')

# Part 1 — Exploratory Data Analysis



**Load the vehicle registration data**

`pd.read_csv()` reads the file into the DataFrame `df`. The path is relative to the notebook kernel’s working directory. `sep=";"` tells pandas that semicolons separate fields, while `decimal=","` handles decimal commas in columns pandas recognizes as numeric. Mixed text and numbers may still require conversion later.


In [2]:
dataset_path = "data/B3/data.csv"
df = pd.read_csv(dataset_path, sep=";", decimal=",")

**Preview the first rows**

`df.head()` displays the first five records. Use this preview to check column names, example values, and whether pandas read the file with the intended separators.


In [ ]:
df.head()

,registration_id,timestamp,country,manufacturer,make,commercial_name,fuel_type,fuel_mode,mass_kg,wltp_test_mass_kg,engine_capacity_cm3,engine_power_kw,electric_consumption_wh_km,electric_range_km,eco_innovation_reduction_g_km,euro_standard,fuel_consumption_l_100km,co2_wltp_g_km,split
0,159461423,2024-01-02,Italy,AUDI AG,AUDI,A3 SPORTBACK,diesel,mono-fuel,1490.00,1606.0,1968,110,NaN,NaN,1.71,EURO 6 AP,4.8,127.0,train
1,159469504,2024-01-02,Italy,CHRYSLER,JEEP,RENEGADE,petrol,hybrid,1495.00,1583.0,1469,96,NaN,NaN,1.35,EURO 6 AP,5.6,127.0,train
2,159515023,2024-01-02,Italy,AUDI AG,AUDI,Q5 SPORTBACK,diesel,hybrid,1900.00,2039.0,1968,150,NaN,NaN,1.86,EURO 6 AP,6.7,176.0,train
3,159681106,2024-01-02,Italy,VOLKSWAGEN,VOLKSWAGEN,GOLF,petrol,hybrid,1.35,1486.0,1498,96,NaN,NaN,NaN,EURO 6 AP,5.6,127.0,train
4,159816114,2024-01-02,Italy,BMW AG,BMW,116D,diesel,mono-fuel,1460.00,1555.0,1496,85,NaN,NaN,1.90,EURO 6 AP,4.6,122.0,train


**Inspect size and column types**

`len(df)` returns the number of rows, and `df.shape` returns `(rows, columns)`. `df.dtypes` shows the type pandas assigned to each column. A date or numeric field stored as text may need conversion before analysis.


In [ ]:
print("Dataset len:\t", len(df))

print("Dataset shape: \t",df.shape)
print("\nDataset dtype:")
df.dtypes

Dataset len: 	 9894
Dataset shape: 	 (9894, 19)

Dataset dtype:


registration_id                    int64
timestamp                            str
country                              str
manufacturer                         str
make                                 str
commercial_name                      str
fuel_type                            str
fuel_mode                            str
mass_kg                          float64
wltp_test_mass_kg                float64
engine_capacity_cm3                int64
engine_power_kw                    int64
electric_consumption_wh_km       float64
electric_range_km                float64
eco_innovation_reduction_g_km    float64
euro_standard                        str
fuel_consumption_l_100km         float64
co2_wltp_g_km                    float64
split                                str
dtype: object

**Count missing values**

`df.isnull()` marks missing entries as `True`. Summing those flags gives the number of missing entries in each column. This detects values pandas treats as missing; text placeholders and implausible numbers need separate checks.


In [ ]:
# Check for missing values in each column
print("Number of missing values per column:")
print(df.isnull().sum())

Number of missing values per column:
registration_id                     0
timestamp                           0
country                             0
manufacturer                        0
make                                0
commercial_name                     0
fuel_type                           0
fuel_mode                           0
mass_kg                             0
wltp_test_mass_kg                  28
engine_capacity_cm3                 0
engine_power_kw                     0
electric_consumption_wh_km       9526
electric_range_km                9526
eco_innovation_reduction_g_km    3030
euro_standard                      29
fuel_consumption_l_100km          164
co2_wltp_g_km                      50
split                               0
dtype: int64


**Summarize numeric columns without the identifier**

`df.drop(columns=["registration_id"])` creates `df_noid`, leaving `df` unchanged. The registration identifier labels records, so numerical summaries of it would not help describe the vehicles. `describe()` summarizes numeric columns by default, including counts, means, standard deviations, quartiles, and extremes. `.round(1)` rounds the displayed statistics to one decimal place, and `.T` puts each variable on its own row. Numeric fields still stored as text are excluded from this summary.


In [ ]:
df_noid = df.drop(columns=["registration_id"])
df_noid.describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
mass_kg,9894.0,1299.2,649.5,1.0,1110.0,1306.0,1537.0,17000.0
wltp_test_mass_kg,9866.0,1498.1,306.7,1046.0,1286.0,1431.0,1672.0,2936.0
engine_capacity_cm3,9894.0,1422.7,455.1,875.0,999.0,1350.0,1598.0,6496.0
engine_power_kw,9894.0,89.4,41.5,-200.0,67.0,81.0,110.0,574.0
electric_consumption_wh_km,368.0,189.5,40.4,123.0,163.0,179.5,207.0,331.0
electric_range_km,368.0,74.9,22.8,16.0,58.0,69.0,84.0,141.0
eco_innovation_reduction_g_km,6864.0,1.5,0.5,0.5,1.0,1.5,1.9,2.5
fuel_consumption_l_100km,9730.0,5.6,1.3,0.4,4.9,5.4,6.0,17.3
co2_wltp_g_km,9844.0,124.9,30.3,7.0,113.0,124.0,137.0,393.0


**Inspect category frequencies**

The loop prints the frequency of each distinct value in six categorical columns. `value_counts()` sorts categories from most to least frequent and excludes missing entries by default. These counts can help reveal rare categories, inconsistent spelling, or labels that may need standardization.


In [ ]:
for col in ["manufacturer", "make", "commercial_name", "fuel_type", "fuel_mode", "euro_standard"]:
    print(df[col].value_counts(), "\n")

manufacturer
STELLANTIS EUROPE             1507
STELLANTIS AUTO                833
VOLKSWAGEN                     763
RENAULT                        640
BMW AG                         546
DACIA                          539
TOYOTA MOTOR CORPORATION       533
AUDI AG                        431
FORD WERKE GMBH                425
MERCEDES-BENZ AG               337
TOYOTA                         306
SKODA                          261
AUTOMOBILES CITROEN            255
SAIC MOTOR CORPORATION         227
NISSAN AUTOMOTIVE EUROPE       221
SEAT                           212
DR AUTOMOBILES                 201
HYUNDAI TURKIYE                189
KIA SLOVAKIA                   165
KIA                            151
ALFA ROMEO                     136
SUZUKI MOTOR CORPORATION       114
MAGYAR SUZUKI                  113
VOLVO                           97
CHRYSLER                        88
MAZDA                           85
HYUNDAI CZECH                   81
AUTOMOBILES PEUGEOT             73
HYUNDAI

**Convert registration dates**

`pd.to_datetime()` converts the `timestamp` column to a pandas datetime type and assigns it back to `df`. This enables date filtering and extracting components such as year or month. With the default settings, an unparseable date raises an error.


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

**Verify the date column type**

Display `df.dtypes` again to check that `timestamp` now has a datetime type. This checks the conversion’s result; it does not establish whether the dates themselves are reasonable.


In [ ]:
print("Dataset dtype after timestamp correction:")
df.dtypes

Dataset dtype after timestamp correction:


registration_id                           int64
timestamp                        datetime64[us]
country                                     str
manufacturer                                str
make                                        str
commercial_name                             str
fuel_type                                   str
fuel_mode                                   str
mass_kg                                 float64
wltp_test_mass_kg                       float64
engine_capacity_cm3                       int64
engine_power_kw                           int64
electric_consumption_wh_km              float64
electric_range_km                       float64
eco_innovation_reduction_g_km           float64
euro_standard                               str
fuel_consumption_l_100km                float64
co2_wltp_g_km                           float64
split                                       str
dtype: object

**Preview the updated data**

`df.head(2)` displays the first two records after the date conversion. Use it to inspect the updated date representation alongside the other fields.


In [ ]:
df.head(2)

,registration_id,timestamp,country,manufacturer,make,commercial_name,fuel_type,fuel_mode,mass_kg,wltp_test_mass_kg,engine_capacity_cm3,engine_power_kw,electric_consumption_wh_km,electric_range_km,eco_innovation_reduction_g_km,euro_standard,fuel_consumption_l_100km,co2_wltp_g_km,split
0,159461423,2024-01-02,Italy,AUDI AG,AUDI,A3 SPORTBACK,diesel,mono-fuel,1490.0,1606.0,1968,110,NaN,NaN,1.71,EURO 6 AP,4.8,127.0,train
1,159469504,2024-01-02,Italy,CHRYSLER,JEEP,RENEGADE,petrol,hybrid,1495.0,1583.0,1469,96,NaN,NaN,1.35,EURO 6 AP,5.6,127.0,train


**Build a data dictionary and check plausible ranges**

The `meaning` dictionary assigns each column a description and optional lower and upper bounds. These bounds are broad assumptions for this exercise. Values outside them deserve investigation before any decision to remove or change them.

`numeric_for_check` prepares temporary numeric versions of two text fields: it removes a trailing `kW` unit from engine power and replaces decimal commas in fuel consumption. `pd.to_numeric(..., errors="coerce")` converts unparseable entries to missing values. These temporary conversions do not overwrite the columns in `df`.

For each column with bounds, the loop selects entries that were nonmissing in the original column and checks whether their numeric values fall within the inclusive range. A failed numeric conversion counts as a failed check. The table reports the observed minimum and maximum, and marks the column `yes` only when all selected entries pass. Originally missing entries are excluded from this range check.

For columns without bounds, the loop reports the number of distinct nonmissing values and labels them `check values`; it does not validate their categories or dates. The `missing %` column uses the original data, so it excludes missing values introduced by the temporary numeric conversions. Finally, `pd.DataFrame(rows)` displays the descriptions, original types, ranges, missing percentages, and check results. No rows are removed by this cell.


In [ ]:
# Broad plausibility bounds for the passenger cars in this exercise.
meaning = {
    "registration_id": ("Identifier of the vehicle registration record", None, None),
    "timestamp": ("Vehicle registration date", None, None),
    "country": ("Country of registration", None, None),
    "manufacturer": ("Vehicle manufacturer as reported in the registration data", None, None),
    "make": ("Vehicle brand or marque", None, None),
    "commercial_name": ("Marketed model or variant name", None, None),
    "fuel_type": ("Reported fuel or fuel combination", None, None),
    "fuel_mode": ("Powertrain fuel mode, such as mono-fuel, bi-fuel, hybrid or plug-in hybrid", None, None),
    "mass_kg": ("Vehicle mass in running order (kg)", 500, 3500),
    "wltp_test_mass_kg": ("Vehicle mass used for WLTP testing (kg)", 500, 4000),
    "engine_capacity_cm3": ("Combustion engine displacement (cm³)", 500, 8500),
    "engine_power_kw": ("Reported engine power (kW)", 20, 800),
    "electric_consumption_wh_km": ("Electric energy consumption (Wh/km)", 50, 500),
    "electric_range_km": ("Reported electric driving range (km)", 1, 250),
    "eco_innovation_reduction_g_km": ("Reported CO₂ savings from eco-innovations (g/km)", 0, 10),
    "euro_standard": ("Euro pollutant-emissions standard or approval stage", None, None),
    "fuel_consumption_l_100km": ("Reported fuel consumption (L/100 km)", 0.1, 25),
    "co2_wltp_g_km": ("WLTP CO₂ emissions (g/km), the regression target", 0, 500),
    "split": ("Prescribed modelling partition (train or test)", None, None),
}

# Parse numeric text locally so the dictionary can check ranges on the raw data.
numeric_for_check = {
    "engine_power_kw": pd.to_numeric(
        df["engine_power_kw"].astype("string").str.replace(r"\s*kW\s*$", "", regex=True),
        errors="coerce",
    ),
    "fuel_consumption_l_100km": pd.to_numeric(
        df["fuel_consumption_l_100km"].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    ),
}

rows = []
for col, (text, lo, hi) in meaning.items():
    s = df[col]
    if lo is not None:
        values = numeric_for_check.get(col, s)[s.notna()]
        valid = values.between(lo, hi).fillna(False)
        observed = f"{values.min():g} to {values.max():g}" if values.notna().any() else "-"
        expected = f"{lo} to {hi}"
        plausible = ("yes" if valid.all() else "no") if len(values) else "no values"
    else:
        observed = f"{s.nunique()} values"
        expected = "-"
        plausible = "check values"

    rows.append({
        "column": col,
        "type": str(s.dtype),
        "meaning": text,
        "expected": expected,
        "observed": observed,
        "missing %": round(s.isna().mean() * 100, 1),
        "plausible?": plausible,
    })

data_dictionary = pd.DataFrame(rows)
data_dictionary


,column,type,meaning,expected,observed,missing %,plausible?
0,registration_id,int64,Identifier of the vehicle registration record,-,9700 values,0.0,check values
1,timestamp,datetime64[us],Vehicle registration date,-,253 values,0.0,check values
2,country,str,Country of registration,-,1 values,0.0,check values
3,manufacturer,str,Vehicle manufacturer as reported in the regist...,-,53 values,0.0,check values
4,make,str,Vehicle brand or marque,-,52 values,0.0,check values
5,commercial_name,str,Marketed model or variant name,-,443 values,0.0,check values
6,fuel_type,str,Reported fuel or fuel combination,-,13 values,0.0,check values
7,fuel_mode,str,"Powertrain fuel mode, such as mono-fuel, bi-fu...",-,4 values,0.0,check values
8,mass_kg,float64,Vehicle mass in running order (kg),500 to 3500,0.991 to 17000,0.0,no
9,wltp_test_mass_kg,float64,Vehicle mass used for WLTP testing (kg),500 to 4000,1046 to 2936,0.3,yes
